# Dummy Input Generation

Este notebook cria os dados de severidade de acidentes, normaliza texto para ASCII, aumenta o volume de exemplos para 80, documenta as features e salva um arquivo `dummy_input.parquet` com fallback para CSV quando o suporte Parquet não estiver disponível.


In [17]:
import pandas as pd
import unicodedata
import random
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


REPO_ROOT = find_repo_root(Path.cwd())
OUTPUT_DIR = REPO_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_PATH = OUTPUT_DIR / "dummy_input.parquet"

print(f"Output directory: {OUTPUT_DIR}")


Output directory: c:\Users\augus\Desktop\UFSC\PPGESE\Disciplinas\Ciencias de Dados\prf-accidents-feature-importance\data\processed


## 1. Creates the first 4 data elements

Criando 4 exemplos de dados de severidade de acidentes com diferentes combinações de características.


In [18]:
base_records = [
    {
        "dia_semana": "Segunda",
        "horario": "manha",
        "condicao_metereologica": "Ceu claro",
        "tipo_pista": "Simples",
        "reta": "Reta",
        "inclinacao": "nivel",
        "volume_pedagio": 1200,
        "icm_via": 2.3,
        "classificacao_acidente": "sem_vitimas",
    },
    {
        "dia_semana": "Quarta",
        "horario": "tarde",
        "condicao_metereologica": "Chuva",
        "tipo_pista": "Dupla",
        "reta": "Curva",
        "inclinacao": "aclive",
        "volume_pedagio": 1800,
        "icm_via": 4.1,
        "classificacao_acidente": "leves",
    },
    {
        "dia_semana": "Sexta",
        "horario": "noite",
        "condicao_metereologica": "Neblina",
        "tipo_pista": "Simples",
        "reta": "Curva",
        "inclinacao": "declive",
        "volume_pedagio": 2500,
        "icm_via": 5.6,
        "classificacao_acidente": "graves_fatais",
    },
    {
        "dia_semana": "Domingo",
        "horario": "madrugada",
        "condicao_metereologica": "Ceu claro",
        "tipo_pista": "Multipla",
        "reta": "Reta",
        "inclinacao": "nivel",
        "volume_pedagio": 3000,
        "icm_via": 1.8,
        "classificacao_acidente": "sem_vitimas",
    },
]

df_original = pd.DataFrame(base_records)
print("Rows:", len(df_original))
print("Columns:", len(df_original.columns))
df_original.head(10)


Rows: 4
Columns: 9


,dia_semana,horario,condicao_metereologica,tipo_pista,reta,inclinacao,volume_pedagio,icm_via,classificacao_acidente
0,Segunda,manha,Ceu claro,Simples,Reta,nivel,1200,2.3,sem_vitimas
1,Quarta,tarde,Chuva,Dupla,Curva,aclive,1800,4.1,leves
2,Sexta,noite,Neblina,Simples,Curva,declive,2500,5.6,graves_fatais
3,Domingo,madrugada,Ceu claro,Multipla,Reta,nivel,3000,1.8,sem_vitimas


## 2. Normalize text columns to ASCII

Normalizamos as colunas categóricas para valores ASCII, removendo acentos de `Manhã`, `Céu claro`, `Múltipla`, etc.


In [19]:
def to_ascii(text: str) -> str:
    normalized = unicodedata.normalize("NFKD", str(text))
    ascii_text = "".join(char for char in normalized if ord(char) < 128)
    return ascii_text

text_columns = [
    "dia_semana",
    "horario",
    "condicao_metereologica",
    "tipo_pista",
    "reta",
]

for column in text_columns:
    df_original[column] = df_original[column].astype(str).map(to_ascii)

print(df_original[text_columns].head(10))


  dia_semana    horario condicao_metereologica tipo_pista   reta
0    Segunda      manha              Ceu claro    Simples   Reta
1     Quarta      tarde                  Chuva      Dupla  Curva
2      Sexta      noite                Neblina    Simples  Curva
3    Domingo  madrugada              Ceu claro   Multipla   Reta


## 3. Increase data with synthetic augmentation

Aumentamos a base usando replicação e pequenas variações nos valores numéricos e categóricos para chegar 2000 linhas.
Vieses são adicionados aos dados para replicar cenários reais de acidentes, como maior severidade em curvas, à noite e em condições meteorológicas adversas.

In [20]:
categorical_options = {
    "dia_semana": ["Segunda", "Terca", "Quarta", "Quinta", "Sexta", "Sabado", "Domingo"],
    "horario": ["manha", "tarde", "noite", "madrugada"],
    "condicao_metereologica": ["Ceu claro", "Chuva", "Neblina", "Neve", "Tempestade"],
    "tipo_pista": ["Simples", "Dupla", "Multipla"],
    "reta": ["Reta", "Curva"],
    "inclinacao": ["nivel", "aclive", "declive"],
    "classificacao_acidente": ["sem_vitimas", "leves", "graves_fatais"],
}

synthetic_rows = []
base_rows = df_original.to_dict(orient="records")

for idx in range(2000):
    template = base_rows[idx % len(base_rows)] # idx % len(base_rows) -> onda periódica (dente de serra) com período 5. 
                                               # À medida que $x$ varia de 1 a 2000, os valores de $y$ alternam continuamente na sequência 1, 2, 3, 4, 0.
    new_row = template.copy()

    new_row["dia_semana"] = categorical_options["dia_semana"][idx % len(categorical_options["dia_semana"])]
    new_row["horario"] = categorical_options["horario"][idx % len(categorical_options["horario"])]
    new_row["condicao_metereologica"] = categorical_options["condicao_metereologica"][idx % len(categorical_options["condicao_metereologica"])]
    new_row["tipo_pista"] = categorical_options["tipo_pista"][idx % len(categorical_options["tipo_pista"])]
    new_row["reta"] = categorical_options["reta"][idx % len(categorical_options["reta"])]

    grave_condition = (
        new_row["tipo_pista"] == "Simples"
        and new_row["condicao_metereologica"] in ["Chuva", "Neblina"]
        and new_row["reta"] == "Curva"
        and new_row["icm_via"] >= 4.0
    )
    low_severity_condition = (
        new_row["volume_pedagio"] >= 2500
        and new_row["tipo_pista"] in ["Dupla", "Multipla"]
        and new_row["condicao_metereologica"] == "Ceu claro"
    )

    if grave_condition:
        new_row["inclinacao"] = "declive"
        new_row["volume_pedagio"] = max(1800, int(new_row["volume_pedagio"] * 1.05))
    elif low_severity_condition:
        new_row["inclinacao"] = "nivel"
        new_row["volume_pedagio"] = int(new_row["volume_pedagio"] * 1.10)
    else:
        new_row["inclinacao"] = categorical_options["inclinacao"][idx % len(categorical_options["inclinacao"])]
        new_row["volume_pedagio"] = max(1000, int(new_row["volume_pedagio"] * 0.95))

    if grave_condition:
        weights = [0.10, 0.20, 0.70]
    elif low_severity_condition:
        weights = [0.70, 0.20, 0.10]
    else:
        weights = [0.35, 0.45, 0.20]

    if new_row["condicao_metereologica"] in ["Chuva", "Neblina"] and new_row["reta"] == "Curva":
        weights = [weights[0] * 0.8, weights[1] * 0.9, weights[2] * 1.2]
    if new_row["volume_pedagio"] >= 2500 and new_row["tipo_pista"] in ["Dupla", "Multipla"]:
        weights = [weights[0] * 1.25, weights[1] * 0.95, weights[2] * 0.7]

    total_weight = sum(weights)
    normalized_weights = [weight / total_weight for weight in weights]
    classification_score = random.choices([0, 1, 2], weights=normalized_weights, k=1)[0]

    if classification_score == 0:
        new_row["classificacao_acidente"] = "sem_vitimas"
    elif classification_score == 1:
        new_row["classificacao_acidente"] = "leves"
    else:
        new_row["classificacao_acidente"] = "graves_fatais"

    synthetic_rows.append(new_row)

    if idx < 5:
        print(new_row)

print("Generated rows:", len(synthetic_rows))

df_augmented = pd.DataFrame(synthetic_rows)
print(df_augmented.shape)
df_augmented.head()


{'dia_semana': 'Segunda', 'horario': 'manha', 'condicao_metereologica': 'Ceu claro', 'tipo_pista': 'Simples', 'reta': 'Reta', 'inclinacao': 'nivel', 'volume_pedagio': 1140, 'icm_via': 2.3, 'classificacao_acidente': 'leves'}
{'dia_semana': 'Terca', 'horario': 'tarde', 'condicao_metereologica': 'Chuva', 'tipo_pista': 'Dupla', 'reta': 'Curva', 'inclinacao': 'aclive', 'volume_pedagio': 1710, 'icm_via': 4.1, 'classificacao_acidente': 'graves_fatais'}
{'dia_semana': 'Quarta', 'horario': 'noite', 'condicao_metereologica': 'Neblina', 'tipo_pista': 'Multipla', 'reta': 'Reta', 'inclinacao': 'declive', 'volume_pedagio': 2375, 'icm_via': 5.6, 'classificacao_acidente': 'sem_vitimas'}
{'dia_semana': 'Quinta', 'horario': 'madrugada', 'condicao_metereologica': 'Neve', 'tipo_pista': 'Simples', 'reta': 'Curva', 'inclinacao': 'nivel', 'volume_pedagio': 2850, 'icm_via': 1.8, 'classificacao_acidente': 'leves'}
{'dia_semana': 'Sexta', 'horario': 'manha', 'condicao_metereologica': 'Tempestade', 'tipo_pista':

,dia_semana,horario,condicao_metereologica,tipo_pista,reta,inclinacao,volume_pedagio,icm_via,classificacao_acidente
0,Segunda,manha,Ceu claro,Simples,Reta,nivel,1140,2.3,leves
1,Terca,tarde,Chuva,Dupla,Curva,aclive,1710,4.1,graves_fatais
2,Quarta,noite,Neblina,Multipla,Reta,declive,2375,5.6,sem_vitimas
3,Quinta,madrugada,Neve,Simples,Curva,nivel,2850,1.8,leves
4,Sexta,manha,Tempestade,Dupla,Reta,aclive,1140,2.3,sem_vitimas


## 4. Save processed data as `dummy_input.parquet`

Verificamos se a dependência PyArrow está disponível e, se estiver, salvamos o dataset em Parquet. Caso contrário, emitimos um alerta no console.

In [21]:
import importlib.util

if importlib.util.find_spec("pyarrow") is None:
    print("ALERTA: pyarrow não está instalado. Instale-o para salvar o arquivo Parquet.")
else:
    df_augmented.to_parquet(PARQUET_PATH, index=False, engine="pyarrow")
    print(f"Saved augmented dataset to {PARQUET_PATH}")


Saved augmented dataset to c:\Users\augus\Desktop\UFSC\PPGESE\Disciplinas\Ciencias de Dados\prf-accidents-feature-importance\data\processed\dummy_input.parquet
